conexión con sentinel-2 y datos


In [1]:
%pip install python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install sentinelhub

  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached pillow-12.3.0-cp311-cp311-win_amd64.whl.metadata (9.3 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
     ---------------------------------------- 0.0/44.9 kB ? eta -:--:--
     ---------------------------------------- 44.9/44.9 kB 2.3 MB/s eta 0:00:00
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/240.4 kB ? eta -:--:--
   ------------------ --------------------- 112.6/240.4 kB 3.3 MB/s eta 0:00:01
   ---------------------------------------  235.5/240.4 kB 2.9 MB/s eta 0:00:01
   ---------------------------------------- 240.4/240.4 kB 2.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/165.6 kB ? eta -:--:--
   


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install rasterio

  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/25.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/25.7 MB 660.6 kB/s eta 0:00:39
   ---------------------------------------- 0.2/25.7 MB 1.5 MB/s eta 0:00:17
   ---------------------------------------- 0.3/25.7 MB 2.2 MB/s eta 0:00:12
    --------------------------------------- 0.5/25.7 MB 3.2 MB/s eta 0:00:09
   - -------------------------------------- 1.1/25.7 MB 4.9 MB/s eta 0:00:06
   -- ------------------------------------- 1.6/25.7 MB 6.2 MB/s eta 0:00:04
   ---- ----------------------------------- 2.6/25.7 MB 8.7 MB/s eta 0:00:03
   ---- ----------------------------------- 2.9/25.7 MB 8.7 MB/s eta 0:00:03
   ----- ---------------------------------- 3.8/25.7 MB 10.2 MB/s eta 0:00:03
   ------- -------------------------------- 4.5/25.7 MB 11.2 MB/s eta 0:00:02
   -------- ---------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
from datetime import datetime

from sentinelhub import (
    SHConfig,
    CRS,
    BBox,
    DataCollection,
    SentinelHubRequest,
    MimeType,
    bbox_to_dimensions
)

# para datos espaciales
import rasterio
from rasterio.transform import from_bounds
import matplotlib.pyplot as plt


In [8]:
import os
from dotenv import load_dotenv
from sentinelhub import SHConfig

load_dotenv()

config = SHConfig()

config.sh_client_id = os.getenv("SH_CLIENT_ID")
config.sh_client_secret = os.getenv("SH_CLIENT_SECRET")

print("Client ID configurado:", bool(config.sh_client_id))
print("Client Secret configurado:", bool(config.sh_client_secret))

Client ID configurado: True
Client Secret configurado: True


#### coordenadas de los lagos

In [9]:
# coordenadas de los lagos que están en el lab 
lagos = {
    'Atitlán': {
        'west': -91.326256 ,
        'east': -91.07151 ,
        'south': 14.5948 ,
        'north': 14.750979
    },

    'Amatitlán': {
        'west': -90.638065 ,
        'east': -90.512924 ,
        'south': 14.412347 ,
        'north': 14.493799
    }
}

# fechas dadas en el lab
fechas_atitlan = [ '2025-01-18', '2025-04-13', '2025-05-13', '2025-07-17', '2025-11-21', '2025-12-29', '2026-02-12', '2026-03-24', '2026-04-13', '2026-04-28', '2026-07-22' ]

fechas_amatitlan = [ '2025-01-28', '2025-04-15', '2025-04-28', '2025-11-24', '2026-01-08', '2026-02-02', '2026-02-07', '2026-03-29', '2026-04-13', '2026-04-28', '2026-06-19' ]

fechas = { 'Atitlán': fechas_atitlan, 'Amatitlán': fechas_amatitlan }

print("Lagos definidos:")
for lago, coords in lagos.items(): print(f"  {lago}: {coords}")
print(f"\nFechas por lago:")
for lago, fecha_list in fechas.items(): print(f"  {lago}: {len(fecha_list)} fechas")

Lagos definidos:
  Atitlán: {'west': -91.326256, 'east': -91.07151, 'south': 14.5948, 'north': 14.750979}
  Amatitlán: {'west': -90.638065, 'east': -90.512924, 'south': 14.412347, 'north': 14.493799}

Fechas por lago:
  Atitlán: 11 fechas
  Amatitlán: 11 fechas


directorio para guardar datos

In [10]:
# Crear carpeta para guardar datos descargados
data_dir = Path('datos_sentinel')
data_dir.mkdir(exist_ok=True)

# Subcarpetas para cada lago
for lago in lagos.keys():
    lago_dir = data_dir / lago
    lago_dir.mkdir(exist_ok=True)

print(f"Directorio de datos: {data_dir.absolute()}")
print(f"Subdirectorios creados para cada lago")

Directorio de datos: c:\Users\darta\OneDrive\Desktop\REN\UVG\2026\Segundo ciclo\Data Science\DataS_Lab4_GeoEspaciales\datos_sentinel
Subdirectorios creados para cada lago


#### ddescargar datos de Sentinel-2

aquí bandas necesarias para calcular 
NDVI: B04 (Rojo) y B08 (NIR - Near Infrared)
NDWI: B03 (Verde) y B08 (NIR)
Cianobacteria



Descarga bandas específicas de Sentinel-2 para un lago en una fecha determinada.
    
    Args:
        bbox (BBox): Caja delimitadora con coordenadas del lago
        fecha (str): Fecha en formato 'YYYY-MM-DD'
        lago_nombre (str): Nombre del lago
        config: Configuración de Sentinel Hub
    
    Returns:
        dict: Diccionario con arrays NumPy de cada banda (B03, B04, B08)

In [13]:
def descargar_bandas_sentinelhub(bbox, fecha, lago_nombre, config=None):

    # las bandas B03 (verde), B04 (rojo), B08 (NIR)
    # SCL para máscara de nubes
    request = SentinelHubRequest(
        evalscript="""
            //VERSION=3
            function setup() {
                return {
                    input: [{
                        bands: ["B03", "B04", "B08", "SCL"],
                        units: "DN"
                    }],
                    output: {
                        bands: 4,
                        sampleType: "FLOAT32"
                    }
                };
            }

            function evaluatePixel(sample) {
                return [sample.B03, sample.B04, sample.B08, sample.SCL];
            }
        """,

        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L2A,
                time_interval=(fecha, fecha),
            )
        ],

        responses=[
            SentinelHubRequest.output_response("default", MimeType.TIFF)
        ],

        bbox=bbox,
        size=bbox_to_dimensions(bbox, resolution=12),
        config=config
    )

    try:
        data = request.get_data()
        print(f"descargado: {lago_nombre} y {fecha}")
        return data[0]

    except Exception as e:
        print(f"error para: {lago_nombre} y {fecha}: {str(e)}")
        return None


print("descarga lista")

descarga lista


In [23]:
import time

datos_descargados = {}

for lago_nombre, coords in lagos.items():

    print("\n" + "=" * 60 )
    print(f"descargando para {lago_nombre}")
    print("=" * 60)

    bbox = BBox(
        [ coords["west"],
            coords["south"],
            coords["east"],
            coords["north"]
        ],
        crs = CRS.WGS84
    )

    datos_descargados[lago_nombre] = {}

    for i, fecha in enumerate(fechas[lago_nombre]):

        print(
            f"\n[{i+1}/{len(fechas[lago_nombre])}] "
            f"descargando {fecha} "
        )

        data = descargar_bandas_sentinelhub(
            bbox,
            fecha,
            lago_nombre,
            config
        )

        if data is not None:
            datos_descargados[lago_nombre][fecha] = data
            print(f"shape: {data.shape}")

        if i < len(fechas[lago_nombre]) - 1: time.sleep(2)


print( "\ndescarga hecha " f"total: {sum(len(v) for v in datos_descargados.values())}" )


descargando para Atitlán

[1/11] descargando 2025-01-18 
descargado: Atitlán y 2025-01-18
shape: (1458, 2275, 4)

[2/11] descargando 2025-04-13 
descargado: Atitlán y 2025-04-13
shape: (1458, 2275, 4)

[3/11] descargando 2025-05-13 
descargado: Atitlán y 2025-05-13
shape: (1458, 2275, 4)

[4/11] descargando 2025-07-17 
descargado: Atitlán y 2025-07-17
shape: (1458, 2275, 4)

[5/11] descargando 2025-11-21 
descargado: Atitlán y 2025-11-21
shape: (1458, 2275, 4)

[6/11] descargando 2025-12-29 
descargado: Atitlán y 2025-12-29
shape: (1458, 2275, 4)

[7/11] descargando 2026-02-12 
descargado: Atitlán y 2026-02-12
shape: (1458, 2275, 4)

[8/11] descargando 2026-03-24 
descargado: Atitlán y 2026-03-24
shape: (1458, 2275, 4)

[9/11] descargando 2026-04-13 
descargado: Atitlán y 2026-04-13
shape: (1458, 2275, 4)

[10/11] descargando 2026-04-28 
descargado: Atitlán y 2026-04-28
shape: (1458, 2275, 4)

[11/11] descargando 2026-07-22 
descargado: Atitlán y 2026-07-22
shape: (1458, 2275, 4)

des

In [ ]:
bandas_procesadas = {}

for lago_nombre, fechas_dict in datos_descargados.items():
    print(f"\guardando  {lago_nombre}"  )

    bandas_procesadas[lago_nombre] = {}

    for fecha, data in fechas_dict.items():

        B03 = data[:, :, 0 ]
        B04 = data[:, :, 1 ]
        B08 = data[:, :, 2 ]
        SCL = data[:, :, 3 ]

        bandas_procesadas[lago_nombre][fecha] = {
            "B03": B03  ,
            "B04": B04 ,
            "B08": B08 ,
            "SCL": SCL
        }

        lago_dir = data_dir / lago_nombre / fecha
        lago_dir.mkdir(exist_ok = True, parents = True)

        np.save(lago_dir / "B03.npy", B03)
        np.save(lago_dir / "B04.npy", B04)
        np.save(lago_dir / "B08.npy", B08)
        np.save(lago_dir / "SCL.npy", SCL)

        print(f"{fecha}  bandas guardadas")

print("\ntodas las bandas ya están guardadas")

<>:4: SyntaxWarning: invalid escape sequence '\g'
<>:4: SyntaxWarning: invalid escape sequence '\g'
C:\Users\belen\AppData\Local\Temp\ipykernel_31724\1387977376.py:4: SyntaxWarning: invalid escape sequence '\g'
  print(f"\guardando  {lago_nombre}")


\guardando  Atitlán
2025-01-18  bandas guardadas
2025-04-13  bandas guardadas
2025-05-13  bandas guardadas
2025-07-17  bandas guardadas
2025-11-21  bandas guardadas
2025-12-29  bandas guardadas
2026-02-12  bandas guardadas
2026-03-24  bandas guardadas
2026-04-13  bandas guardadas
2026-04-28  bandas guardadas
2026-07-22  bandas guardadas
\guardando  Amatitlán
2025-01-28  bandas guardadas
2025-04-15  bandas guardadas
2025-04-28  bandas guardadas
2025-11-24  bandas guardadas
2026-01-08  bandas guardadas
2026-02-02  bandas guardadas
2026-02-07  bandas guardadas
2026-03-29  bandas guardadas
2026-04-13  bandas guardadas
2026-04-28  bandas guardadas
2026-06-19  bandas guardadas

todas las bandas ya están guardadas
